In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.2.2
tiktoken version: 0.12.0


In [2]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:199])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married 


In [4]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

result = re.split(r'([,.]|\s)', text)

print(result)
##################
#or [,.\s]
result = [item for item in result if item.strip()]
print(result)
################
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

##############

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']
['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']
['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']
['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [5]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
4690


In [6]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(all_words)
print(vocab_size)

['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin', 'Burlington', 'But', 'By', 'Carlo', 'Chicago', 'Claude', 'Come', 'Croft', 'Destroyed', 'Devonshire', 'Don', 'Dubarry', 'Emperors', 'Florence', 'For', 'Gallery', 'Gideon', 'Gisburn', 'Gisburns', 'Grafton', 'Greek', 'Grindle', 'Grindles', 'HAD', 'Had', 'Hang', 'Has', 'He', 'Her', 'Hermia', 'His', 'How', 'I', 'If', 'In', 'It', 'Jack', 'Jove', 'Just', 'Lord', 'Made', 'Miss', 'Money', 'Monte', 'Moon-dancers', 'Mr', 'Mrs', 'My', 'Never', 'No', 'Now', 'Nutley', 'Of', 'Oh', 'On', 'Once', 'Only', 'Or', 'Perhaps', 'Poor', 'Professional', 'Renaissance', 'Rickham', 'Riviera', 'Rome', 'Russian', 'Sevres', 'She', 'Stroud', 'Strouds', 'Suddenly', 'That', 'The', 'Then', 'There', 'They', 'This', 'Those', 'Though', 'Thwing', 'Thwings', 'To', 'Usually', 'Venetian', 'Victor', 'Was', 'We', 'Well', 'What', 'When', 'Why', 'Yes', 'You', '_', 'a', 'abdication', 'able', 'about', 'above',

In [7]:
all_words_test = ['hello', 'world', 'hello', 'this', 'is', 'a', 'test']
vocab_test = {token: integer for integer, token in enumerate(all_words_test)}
#print(list(enumerate(all_words_test)))
print(vocab_test)
#print(list(enumerate(vocab_test)))
for value in vocab_test.values():
	print(value)
for key in vocab_test.keys():
	print(key)
for s,i in vocab_test.items():
	print(s)
	print(i)

{'hello': 2, 'world': 1, 'this': 3, 'is': 4, 'a': 5, 'test': 6}
2
1
3
4
5
6
hello
world
this
is
a
test
hello
2
world
1
this
3
is
4
a
5
test
6


In [8]:
vocab = {token:integer for integer,token in enumerate(all_words)}
#vocab = {}
#for i, token in enumerate(all_words):
#    if token not in vocab:
#        vocab[token] = len(vocab)  # Sequential IDs: 0,1,2,...
print(len(vocab))

1130


In [9]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 5:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)


In [10]:
print(list(enumerate(vocab.items())))

[(0, ('!', 0)), (1, ('"', 1)), (2, ("'", 2)), (3, ('(', 3)), (4, (')', 4)), (5, (',', 5)), (6, ('--', 6)), (7, ('.', 7)), (8, (':', 8)), (9, (';', 9)), (10, ('?', 10)), (11, ('A', 11)), (12, ('Ah', 12)), (13, ('Among', 13)), (14, ('And', 14)), (15, ('Are', 15)), (16, ('Arrt', 16)), (17, ('As', 17)), (18, ('At', 18)), (19, ('Be', 19)), (20, ('Begin', 20)), (21, ('Burlington', 21)), (22, ('But', 22)), (23, ('By', 23)), (24, ('Carlo', 24)), (25, ('Chicago', 25)), (26, ('Claude', 26)), (27, ('Come', 27)), (28, ('Croft', 28)), (29, ('Destroyed', 29)), (30, ('Devonshire', 30)), (31, ('Don', 31)), (32, ('Dubarry', 32)), (33, ('Emperors', 33)), (34, ('Florence', 34)), (35, ('For', 35)), (36, ('Gallery', 36)), (37, ('Gideon', 37)), (38, ('Gisburn', 38)), (39, ('Gisburns', 39)), (40, ('Grafton', 40)), (41, ('Greek', 41)), (42, ('Grindle', 42)), (43, ('Grindles', 43)), (44, ('HAD', 44)), (45, ('Had', 45)), (46, ('Hang', 46)), (47, ('Has', 47)), (48, ('He', 48)), (49, ('Her', 49)), (50, ('Hermia',

In [11]:
import re

class SimpleTokenizerV1:
	def __init__(self,vocab):
		self.str_to_int = vocab
		self.int_to_str = {i:s for s,i in vocab.items()}
	
	def encode(self,text):
		preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
		preprocessed = [item.strip() for item in preprocessed if item.strip()]
		ids = [self.str_to_int[s] for s in preprocessed]
		return ids

	def decode(self,ids):
		text = " ".join([self.int_to_str[i] for i in ids])
		text = re.sub(r'\s+([,.?!"()\'])', r'\1',text)
		return text


In [12]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
s = [tokenizer.int_to_str[i] for i in ids]
#print(ids)
#print(s)
tokenizer.decode(ids)
#print(tokenizer.decode(ids))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [13]:
all_tokens = sorted(list(set(preprocessed)))
#print(len(all_tokens))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
#print(len(all_tokens))
#print(all_tokens)
vocab = {token:integer for integer,token in enumerate(all_tokens)}
#print(vocab)
print(len(vocab.items()))
print(list(vocab.items())[-5:])
#for i, item in enumerate(list(vocab.items())[-5:]):
#   print(item)

1132
[('younger', 1127), ('your', 1128), ('yourself', 1129), ('<|endoftext|>', 1130), ('<|unk|>', 1131)]


In [14]:
import re

class SimpleTokenizerV2:
	def __init__(self, vocab):
		self.str_to_int = vocab
		self.int_to_str = {i:s for s,i in vocab.items()}

	def encode(self,text):
		preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
		preprocessed = [item.strip() for item in preprocessed if item.strip()]
		preprocessed = [
			item if item in self.str_to_int 
			else "<|unk|>" for item in preprocessed
			]
		ids = [self.str_to_int[s] for s in preprocessed]
		return ids

	def decode(self,ids):
		text = " ".join([self.int_to_str[i] for i in ids])
		text = re.sub(r'\s+([,.:;?!"()\'])',r'\1',text)
		return text

In [15]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

tokenizer.encode(text)
tokenizer.decode(tokenizer.encode(text))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

In [16]:
import importlib
import tiktoken
print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [17]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
text = ("Akwirw ier")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

strings = tokenizer.decode(integers)

print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


In [18]:
with open("the-verdict.txt", 'r', encoding="utf-8") as f:
	raw_tex = f.read()
#print(raw_text)
enc_text = tokenizer.encode(raw_text)
#print(len(enc_text))

enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

for i in range(context_size):
	context = enc_sample[:i+1]
	desired = enc_sample[i+1]
	print(context, "---->", desired)

for i in range(context_size):
	context = enc_sample[:i+1]
	desired = enc_sample[i+1]
	print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]
[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257
 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [19]:
#import torch
#print("PyTorch version:", torch.__version__)
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

In [20]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
	def __init__(self, txt, tokenizer, max_length, stride):
		self.input_ids = []
		self.target_ids = []

		# Tokenize the entire text
		token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
		assert len(token_ids) > max_length,  "Number of tokenized inputs must at least be equal to max_length+1"

	# Use a sliding window to chunk the book into overlapping sequences of max_length
		for i in range(0,len(token_ids) - max_length, stride):
			input_chunk = token_ids[i:i+max_length]
			target_chunk = token_ids[i+1:i+max_length+1] 
			self.input_ids.append(torch.tensor(input_chunk))
			self.target_ids.append(torch.tensor(target_chunk))

	def __len__(self):
		return len(self.input_ids)
	
	def __getitem__(self, index):
		return self.input_ids[index], self.target_ids[index]

In [21]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
	tokenizer = tiktoken.get_encoding("gpt2")

	dataset = GPTDatasetV1(txt,tokenizer,max_length=max_length,stride=stride)

	dataloader = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)

	return dataloader

In [22]:
with open("the-verdict.txt", 'r', encoding="utf-8") as f:
	raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4,stride=1,shuffle= False)
data_iter=iter(dataloader)
#print(data_iter)
first_batch= next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [23]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
#print(f"Inputs:\n {inputs}")

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [24]:
input_ids = torch.tensor([2,3,5,1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
print(embedding_layer.weight)
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(input_ids))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [ ]:
max_length = 2

dataloader = create_dataloader_v1(raw_text, batch_size=3, max_length=max_length, stride=max_length, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)
print(len(inputs))

##########################################
vocab_size = 5000
output_dim = 8

#vocab_size = 128000
#output_dim = 32

token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)
print(token_embeddings)

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length,output_dim)
pos_embedding = pos_embedding_layer(torch.arange(context_length))
print(pos_embedding.shape)
print(pos_embedding)

input_embeddings = token_embeddings+pos_embedding
#strange adding together. modern architectures use ALiBi and RoPE
print(input_embeddings.shape)
print(input_embeddings)



Token IDs:
 tensor([[  40,  367],
        [2885, 1464],
        [1807, 3619]])

Inputs shape:
 torch.Size([3, 2])
3
torch.Size([3, 2, 8])
tensor([[[ 1.0560,  0.9128, -0.0718,  0.5295,  0.2891,  0.9421,  0.6838,
          -0.5438],
         [-0.7046,  0.2403, -0.3468,  1.2080, -0.9874,  0.9155, -0.2198,
          -0.2322]],

        [[-0.4406, -0.2491, -0.0554,  1.3980, -0.7198,  0.1529, -0.5504,
          -0.6070],
         [-0.9024, -0.3211,  1.2483,  0.4620, -1.3754, -0.4937, -1.0217,
           0.9982]],

        [[ 1.3087, -0.8542, -0.6132,  0.5841, -0.6478,  0.6834, -1.2265,
           0.3855],
         [ 1.1102,  0.3198, -1.4194, -0.7397, -0.8364, -0.7086,  1.8606,
          -1.0584]]], grad_fn=<EmbeddingBackward0>)
torch.Size([2, 8])
tensor([[ 0.9135, -2.1581,  1.2084, -1.4662,  0.5261,  0.5609, -0.4254,  1.8473],
        [ 0.5198,  0.7860, -0.6902, -1.4829,  0.5754,  0.6497,  0.2628, -0.6494]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([3, 2, 8])
tensor([[[ 1.9695, -1.245